# **ITCC508 Lab Exercise PT-M1 (Building and Evaluating a Domain-Specific RAG Chatbot)**

## Environment Setup & Package Installation

In this step, we install all the Python libraries needed to build the RAG chatbot.

This is necessary because the chatbot needs different tools to perform each part of the RAG process. RAG means **Retrieval-Augmented Generation**. This means the chatbot will first retrieve related information from the uploaded documents before generating an answer.

Line-by-line explanation:

- `!pip install -q ...` tells Google Colab or Jupyter Notebook to install the required packages.
- `langchain` is the main framework used to connect the parts of the RAG system.
- `langchain-community` provides community tools such as document loaders, which help load files like PDF and TXT documents.
- `langchain-groq` allows the notebook to connect to Groq’s language model.
- `langchain-huggingface` allows us to use Hugging Face embedding models.
- `chromadb` is the vector database where document embeddings will be stored.
- `pypdf` helps read and extract text from PDF files.
- `unstructured` gives additional document processing support for different file formats.

This step is important because without these libraries, the notebook will not be able to load documents, create embeddings, store vectors, retrieve relevant chunks, or generate chatbot responses.

In [1]:
# Cell 1: Environment Setup & Package Installation
!pip install -q langchain langchain-community langchain-groq langchain-huggingface chromadb pypdf unstructured

## Secure API Key Configuration

In this step, we securely configure the Groq API key.

The Groq API key is needed because our chatbot will use the `openai/gpt-oss-120b` model through Groq. Instead of typing the API key directly into the code, we use `getpass` so that the key will be hidden while typing.

Line-by-line explanation:

- `import os` imports the operating system module. This is used to access and store environment variables.
- `import getpass` imports a secure password input tool. It allows the user to type the API key without showing it on the screen.
- `if "GROQ_API_KEY" not in os.environ:` checks if the Groq API key is already stored in the notebook environment.
- `os.environ["GROQ_API_KEY"] = getpass.getpass(...)` asks the user to enter the Groq API key securely.
- The API key is stored temporarily in the notebook runtime environment.
- This is safer than writing the API key directly in the notebook because it prevents the key from being exposed in the code.

This step is necessary because the language model cannot run without authentication from Groq.

In [2]:
# Cell 2: Secure API Key Configuration
import os
import getpass

# Prompts for API key securely without saving or echoing plain text in notebook cells
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


## Loading Custom Data & Chunking

In this step, we load the uploaded PDF files from the `my_data` folder and split them into smaller text chunks.

This is one of the most important steps in RAG because the chatbot cannot efficiently search one whole PDF at once. Instead, the documents must be divided into smaller, manageable parts called chunks. These chunks will later be converted into embeddings and stored in a vector database.

Line-by-line explanation:

- `from langchain_community.document_loaders import DirectoryLoader` imports `DirectoryLoader`, which is used to load all files inside a directory.
- `from langchain_text_splitters import RecursiveCharacterTextSplitter` imports a text splitter that divides long documents into smaller chunks.
- `os.makedirs("my_data", exist_ok=True)` creates a folder named `my_data` if it does not already exist.
- `exist_ok=True` prevents an error if the folder already exists.
- `loader = DirectoryLoader(...)` creates a loader that will read all files inside the `my_data` folder.
- `"my_data/"` tells the loader where the files are located.
- `glob="**/*.*"` means the loader will search for all files in the folder and subfolders.
- `show_progress=True` displays loading progress while documents are being processed.
- `raw_documents = loader.load()` loads the content of the files into LangChain document objects.
- `text_splitter = RecursiveCharacterTextSplitter(...)` creates the tool that will split the loaded documents into smaller chunks.
- `chunk_size=500` means each chunk will contain around 500 characters.
- `chunk_overlap=50` means each chunk will overlap with the next chunk by 50 characters.
- The overlap is important because it helps preserve meaning between chunks. Without overlap, some important context may be cut off.
- `documents = text_splitter.split_documents(raw_documents)` applies the splitter to the loaded documents.
- `print(...)` displays how many raw documents were loaded and how many chunks were created.

This step is necessary because RAG works best when the source documents are broken into searchable pieces. The retriever will later use these chunks to find the most relevant information for the user’s question.

In [4]:
# Cell 3: Loading Custom Data & Chunking
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create target data directory
os.makedirs("my_data", exist_ok=True)

# Load all documents from the directory
loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)
raw_documents = loader.load()

# Split documents into smaller semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
documents = text_splitter.split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")

100%|██████████| 2/2 [01:00<00:00, 30.44s/it]


Loaded 2 raw document(s) and split into 2491 chunks.


### PDF Processing Dependency

This additional setup cell installs the PDF-processing dependency required by the notebook. It is necessary for the document loader to successfully process the PDF files used as the chatbot's knowledge source.

In [3]:
!pip install "unstructured[pdf]"

##  Embedding Model & Vector DB Indexing

In this step, we convert the document chunks into embeddings and store them in ChromaDB.

Embeddings are numerical representations of text. They allow the computer to understand the meaning of the document chunks and compare them with the meaning of a user’s question.

Line-by-line explanation:

- `from langchain_huggingface import HuggingFaceEmbeddings` imports the Hugging Face embedding tool.
- `from langchain_community.vectorstores import Chroma` imports Chroma, which will be used as the vector database.
- `embeddings = HuggingFaceEmbeddings(...)` initializes the embedding model.
- `model_name="all-MiniLM-L6-v2"` selects the embedding model that will convert text into numerical vectors.
- This model is lightweight and commonly used for semantic search tasks.
- `vectorstore = Chroma.from_documents(...)` creates a Chroma vector database from the document chunks.
- `documents=documents` tells Chroma to use the chunks created in the previous step.
- `embedding=embeddings` tells Chroma which embedding model to use.
- Chroma stores the vector form of each chunk so that the system can quickly search for relevant information.
- `retriever = vectorstore.as_retriever(...)` converts the vector database into a retriever.
- `search_kwargs={"k": 3}` means the retriever will return the top 3 most relevant chunks for each user query.

This step is necessary because the chatbot needs a way to search the uploaded documents based on meaning, not just exact keywords. The retriever is the component that finds the most relevant chunks before the language model generates an answer.

In [5]:
# Cell 4: Embedding Model & Vector DB Indexing
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store embeddings into Chroma vector database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# Set vectorstore as a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Model Initialization and Domain System Prompt

In this step, we initialize the language model and create the system prompt that controls how the chatbot answers.

The language model is responsible for generating the final response. However, since this is a RAG chatbot, the model should not answer freely using general knowledge. It must answer only from the retrieved context.

Line-by-line explanation:

- `from langchain_groq import ChatGroq` imports the Groq chat model wrapper.
- `from langchain_core.prompts import ChatPromptTemplate` imports the prompt template tool.
- `llm = ChatGroq(...)` creates the language model object.
- `model_name="openai/gpt-oss-120b"` tells the system to use the GPT-OSS 120B model through Groq.
- `temperature=0` makes the model more deterministic and less creative.
- A lower temperature is important in RAG because we want factual and grounded answers.
- `system_prompt = (...)` creates the instruction that the chatbot must follow.
- `"You are a specialized AI assistant..."` tells the model its role.
- `"Answer questions strictly using ONLY the provided context below."` prevents the model from using unsupported outside information.
- `"If the answer cannot be found..."` provides the fallback response for missing information.
- `"Context:\n{context}"` is where the retrieved document chunks will be inserted.
- `prompt = ChatPromptTemplate.from_messages(...)` creates a chat-style prompt.
- `("system", system_prompt)` gives the model its rules and retrieved context.
- `("human", "{input}")` represents the user’s question.

This step is necessary because the prompt is what keeps the chatbot grounded. Without a strict system prompt, the model may hallucinate or invent answers that are not supported by the documents.

In [6]:
# Cell 5: Model Initialization and Domain System Prompt
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the SLM
llm = ChatGroq(
    model_name="openai/gpt-oss-120b",
    temperature=0
)

# Custom domain system prompt
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    "If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

## Pipeline Assembly

In this step, we connect the retriever, prompt, and language model into one complete RAG chain.

This is the stage where the separate parts of the system become a working chatbot pipeline.

Line-by-line explanation:

- `from langchain.chains import create_retrieval_chain` imports a function that creates a retrieval-based chain.
- `from langchain.chains.combine_documents import create_stuff_documents_chain` imports a function that combines retrieved documents and sends them to the model.
- `combine_docs_chain = create_stuff_documents_chain(llm, prompt)` connects the language model with the prompt.
- This chain receives the retrieved context and the user’s question, then generates an answer.
- The word “stuff” means the retrieved document chunks are inserted directly into the prompt.
- `rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` creates the complete RAG pipeline.
- The retriever first searches the vector database for relevant chunks.
- The retrieved chunks are then passed into the document-combining chain.
- The language model finally generates an answer using only the retrieved context.

This step is necessary because RAG requires both retrieval and generation. The retriever finds relevant information, while the language model uses that information to produce the final response.

In [8]:
# Cell 6: Pipeline Assembly
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Combine prompt and LLM to process context
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

### LangChain Version Setup

This additional setup cell installs the LangChain package versions required by the notebook. It helps ensure compatibility among the LangChain components used to load, process, retrieve, and generate responses from the documents.

In [7]:
!pip install "langchain==0.3.27" "langchain-community==0.3.27"

## Testing the Custom Domain Chatbot

In this step, we test the chatbot using an in-domain question.

Since our chosen domain is Lifelong Learning, which is aligned with SDG 4: Quality Education, we ask a question that the uploaded documents are expected to answer.

Line-by-line explanation:

- `user_query = "Summarize the key information found in the documents."` stores the test question in a variable.
- `response = rag_chain.invoke({"input": user_query})` sends the question to the RAG chain.
- The RAG chain retrieves relevant chunks from ChromaDB.
- The retrieved chunks are inserted into the prompt.
- The language model generates an answer based on those chunks.
- `print("--- DOMAIN QUERY ANSWER ---")` prints a label for the chatbot answer.
- `print(response["answer"])` prints the final answer generated by the chatbot.
- `print("\n--- RETRIEVED SOURCE CHUNKS ---")` prints a label for the retrieved sources.
- `for i, doc in enumerate(response["context"]):` loops through the retrieved document chunks.
- `enumerate(...)` gives each retrieved chunk a number.
- `doc.metadata.get("source", "Unknown")` gets the filename or source path of each retrieved chunk.
- `"Unknown"` is used if the source metadata is missing.
- `print(...)` displays the source of each retrieved chunk.

This step is necessary because it verifies whether the chatbot can retrieve relevant information from the uploaded documents and generate a grounded answer.

In [9]:
# Cell 7: Testing Your Custom Domain Chatbot

# Test Case 1: In-Domain Query
user_query = "Summarize the key information found in the documents."
response = rag_chain.invoke({"input": user_query})

print("--- DOMAIN QUERY ANSWER ---")
print(response["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
The provided excerpt gives only a brief overview of the report’s metadata and purpose:

- **Purpose/Scope**: The report’s structure is presented as something that could be considered a “lll initiative,” and the author is asked to review the document and provide a critical evaluation.  
- **Production Credits**:  
  - **Copy‑editing**: Alec McAulay  
  - **Design**: Christiane Marwecki  
- **Acknowledgment**: The author expresses gratitude to both the copy‑editor and the designer for their hard work.  

No further substantive content or findings from the report are included in the supplied text.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/381666eng.pdf
Chunk 2 Source: my_data/UIL_Handbook_170x240_RZ220601_Online.pdf
Chunk 3 Source: my_data/381666eng.pdf


## Task 1: Domain & Dataset Description

### Chosen Domain

**Lifelong Learning**

### Problem Statement

The problem addressed by this project is the difficulty of finding relevant information about lifelong learning from multiple documents. This project aims to develop a domain-specific RAG chatbot that can retrieve relevant information from the provided documents and use it to answer questions about lifelong learning.

### Dataset Description

The dataset used for this RAG chatbot consists of two PDF documents related to lifelong learning, adult learning, and education. The documents provide information about lifelong learning, adult education, continuing education, citizenship education, participation, inclusion, and skills development.

The two PDF files used as the knowledge base are:

1. `381666eng.pdf`
2. `UIL_Handbook_170x240_RZ220601_Online.pdf`

These documents were placed in the `my_data/` folder and loaded using the `DirectoryLoader`. The documents were then divided into smaller chunks using the `RecursiveCharacterTextSplitter` with a chunk size of 500 and an overlap of 50 characters.

The system successfully loaded **2 raw documents** and divided them into **2,491 chunks**. The resulting chunks serve as the knowledge base of the RAG chatbot. They are converted into embeddings and stored in a vector database, allowing the retriever to find relevant information when a user asks a question about lifelong learning.

### Dataset Source

The dataset consists of the two PDF documents listed above, which provide information relevant to the selected **Lifelong Learning** domain.

## Task 2: Out-of-Domain Fallback Test

This code tests whether the RAG chatbot can correctly handle a question that is outside the selected Lifelong Learning domain.

- `user_query = ...` stores the out-of-domain question that will be given to the chatbot. The question is about the solar system, which is not part of the uploaded Lifelong Learning documents.
- `response = rag_chain.invoke({"input": user_query})` sends the question to the RAG chain. The retriever first searches the vector database for relevant document chunks, and the retrieved context is then provided to the language model.
- `print("--- OUT-OF-DOMAIN QUERY ---")` displays a label so the test question is easy to identify in the notebook output.
- `print(user_query)` displays the exact question used for the experiment.
- `print("\n--- CHATBOT OUTPUT ---")` displays a label for the chatbot's response.
- `print(response["answer"])` extracts and displays the final answer generated by the RAG chatbot.

This test is necessary because a domain-specific RAG system should avoid providing unsupported information when the requested answer cannot be found in the provided domain data.

In [10]:
# Task 2: Out-of-Domain Fallback Test

user_query = "What are the planets in the solar system and how do they orbit the sun?"

response = rag_chain.invoke({"input": user_query})

print("--- OUT-OF-DOMAIN QUERY ---")
print(user_query)

print("\n--- CHATBOT OUTPUT ---")
print(response["answer"])

--- OUT-OF-DOMAIN QUERY ---
What are the planets in the solar system and how do they orbit the sun?

--- CHATBOT OUTPUT ---
I cannot answer based on the provided domain data.


## Task 3: Hallucination Stress-Test

**Step 1: Modify the LLM Configuration**

In this step, the language model configuration is intentionally changed to make the chatbot less controlled for the hallucination stress-test.

- `from langchain_groq import ChatGroq` imports the Groq chat model interface used to connect the RAG system to the selected language model.
- `from langchain_core.prompts import ChatPromptTemplate` imports the class used to create the structured system and user prompt.
- `llm = ChatGroq(...)` initializes the same language model used in the experiment.
- `model_name="openai/gpt-oss-120b"` specifies the model used for the experiment.
- `temperature=1.0` increases the model's randomness compared with the original `temperature=0` setting. This allows us to observe whether the model becomes less predictable during the stress-test.
- `system_prompt = (...)` defines the instructions given to the language model.
- The instruction `Answer questions strictly using ONLY the provided context below.` keeps the model grounded in the retrieved RAG context.
- The explicit fallback instruction from the original configuration is intentionally removed. This allows us to observe how the model responds when the requested information is not available.
- `{context}` is a placeholder where the retrieved document chunks will be inserted by the RAG chain.
- `ChatPromptTemplate.from_messages(...)` creates the final prompt structure containing a system message and the user's input.
- `("system", system_prompt)` provides the model's instructions.
- `("human", "{input}")` provides the user's question.

This step is necessary for comparing the original controlled configuration with a less constrained configuration and observing whether the model produces unsupported information.

In [11]:
# Task 3: Modified Cell 5 - Hallucination Stress-Test

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the same SLM with higher temperature
llm = ChatGroq(
    model_name="openai/gpt-oss-120b",
    temperature=1.0
)

# Modified system prompt without the explicit fallback instruction
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

### Rebuilding the RAG Chain

After changing the language model temperature and system prompt, the RAG chain must be rebuilt so that the stress-test uses the modified configuration.

- `create_retrieval_chain` is imported to create the retrieval-based pipeline that connects the retriever with the document-processing chain.
- `create_stuff_documents_chain` is imported to combine the retrieved document chunks with the prompt before sending them to the language model.
- `combine_docs_chain = create_stuff_documents_chain(llm, prompt)` creates a document-combining chain using the modified LLM and prompt.
- `rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` connects the existing retriever to the newly configured document-combining chain.
- `print(...)` confirms that the RAG chain has been successfully rebuilt.

Rebuilding the chain is necessary because the previous RAG chain was created using the original model and prompt configuration. The rebuilt chain ensures that the next test uses `temperature=1.0` and the modified system prompt.

In [12]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("RAG chain rebuilt for hallucination stress-test.")

RAG chain rebuilt for hallucination stress-test.


### Step 2: Re-run the Same Out-of-Domain Question

The exact same question from Task 2 is used again so that the two configurations can be compared fairly.

- `user_query = ...` stores the same out-of-domain question used in Task 2.
- `response = rag_chain.invoke({"input": user_query})` sends the question through the rebuilt RAG chain using the modified model configuration.
- The retriever searches the vector database for relevant context before the language model generates its response.
- `print("--- HALLUCINATION STRESS-TEST QUERY ---")` identifies this test in the notebook output.
- `print(user_query)` displays the exact question used.
- `print("\n--- CHATBOT RESPONSE ---")` identifies the generated response.
- `print(response["answer"])` displays the chatbot's final answer.

Using the same question is important because it allows the effect of the changed temperature and prompt instructions to be observed without changing the test query.

In [13]:
# Task 3: Re-run the same out-of-domain query

user_query = "What are the planets in the solar system and how do they orbit the sun?"
response = rag_chain.invoke({"input": user_query})

print("--- HALLUCINATION STRESS-TEST QUERY ---")
print(user_query)

print("\n--- CHATBOT RESPONSE ---")
print(response["answer"])

--- HALLUCINATION STRESS-TEST QUERY ---
What are the planets in the solar system and how do they orbit the sun?

--- CHATBOT RESPONSE ---
I’m sorry, but the information you’re asking for isn’t included in the provided context.


### Step 3: Observe & Compare

**Original Configuration**

- Temperature: 0
- Fallback: Enabled
- Response: “I cannot answer based on the provided domain data.”

**Response:**

“I cannot answer based on the provided domain data.”

#### Hallucination Stress-Test Configuration

The stress-test configuration used `temperature=1.0` and removed the explicit fallback instruction while keeping the same out-of-domain question and context-only instruction.

**Response:**

“I’m sorry, but the information you’re asking for isn’t included in the provided context.”

#### Comparison

The two responses differed in wording, but both avoided providing unsupported information about the planets or the solar system. The original configuration followed the explicit fallback instruction, producing the predefined response.

In the stress-test configuration, the temperature was increased to 1.0 and the explicit fallback instruction was removed. The model still recognized that the requested information was not available in the provided context and refused to provide an unsupported answer.

The main difference was response control. The original configuration produced the exact predefined fallback message, while the stress-test configuration generated a different refusal. No hallucination was observed in this particular test.

This result shows that increasing the temperature does not necessarily produce hallucination in every test. However, removing the explicit fallback instruction makes the response less predictable, making the configuration useful for stress-testing the reliability of the RAG system.

## Additional Demonstration: Lifelong Learning RAG Chatbot Interface

This section provides a simple terminal-style interface for interacting with the completed Lifelong Learning RAG chatbot.

The interface allows the user to enter questions related to Lifelong Learning. Each question is passed through the RAG chain, which retrieves relevant information from the uploaded documents before generating the chatbot's response.

This is an additional demonstration of the working chatbot interface and is not part of the required Task 1–3 experiments.

### Code Explanation

- `print(...)` displays the chatbot title and usage instructions.
- `while True:` keeps the chatbot running so the user can ask multiple questions.
- `user_query = input(...)` accepts the user's question.
- `if user_query.lower() == "exit":` checks whether the user wants to end the chatbot.
- `rag_chain.invoke({"input": user_query})` sends the question through the RAG pipeline for retrieval and answer generation.
- `print(...)` displays the chatbot's response.
- `break` stops the loop when the user enters `exit`.

This interface is useful because it demonstrates the completed RAG chatbot in an interactive terminal-style format.

In [14]:
# Simple terminal-style Lifelong Learning RAG Chatbot

print("=" * 45)
print("       LIFELONG LEARNING RAG CHATBOT")
print("=" * 45)
print("\nAsk questions about Lifelong Learning.")
print("Type 'exit' to end the chatbot.\n")

while True:
    user_query = input("You: ")

    if user_query.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    response = rag_chain.invoke({"input": user_query})

    print("\nChatbot:", response["answer"])
    print()

       LIFELONG LEARNING RAG CHATBOT

Ask questions about Lifelong Learning.
Type 'exit' to end the chatbot.

You:  What does SDG 4 call on countries to do?   

Chatbot: SDG 4 calls on countries to adopt **sound, evidence‑based and inclusive policies** that go beyond earlier education commitments. Specifically, it urges governments to:

- Prioritise **early‑childhood education**.  
- Ensure **universal completion of primary and secondary schooling**.  
- Provide **equal access to post‑secondary education** for all.  
- **Include marginalized groups** in all levels of education.  
- Focus on **relevant learning outcomes**, building foundational skills and competencies that support sustainable development.

You: What are flexible learning pathways in lifelong learning?  

Chatbot: Flexible learning pathways are **structured routes within the lifelong‑learning system that give learners multiple, adaptable options to acquire, validate, and build on knowledge and skills**.  

- They **promo